# Dhvani: Google Colab F5-TTS experiment

Use **Runtime → Change runtime type → T4 GPU** before running. Colab's free GPU is temporary and may be unavailable. This notebook never uploads audio automatically.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/karthik7026/dhvani-kannada-tts.git /content/dhvani
%cd /content/dhvani
!pip -q install -r requirements.txt
!git clone --depth 1 https://github.com/SWivid/F5-TTS.git /content/F5-TTS
!pip -q install -e /content/F5-TTS

Put only your reviewed private voice dataset in Google Drive at `MyDrive/dhvani-training/data/`, containing `metadata.csv` and `wavs/`. Every clip needs its exact transcript and status `approved`. Do not use a shared or public Drive folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess
data_dir = Path('/content/drive/MyDrive/dhvani-training/data')
assert (data_dir / 'metadata.csv').is_file(), 'Add the reviewed private dataset to Drive first.'
subprocess.run(['python', 'training/export_f5_manifest.py', str(data_dir), '/content/metadata_f5.csv'], check=True)
print('Reviewed manifest is valid.')

In [ ]:
%cd /content/F5-TTS
!python src/f5_tts/train/datasets/prepare_csv_wavs.py /content/metadata_f5.csv data/dhvani_kn_custom --pretrain
# Conservative batch size for a free T4-class GPU.
!f5-tts_finetune-cli --exp_name F5TTS_v1_Base --dataset_name dhvani_kn_custom --tokenizer custom --tokenizer_path data/dhvani_kn_custom/vocab.txt --finetune --epochs 10 --batch_size_per_gpu 400 --batch_size_type frame --max_samples 16 --grad_accumulation_steps 8 --save_per_updates 250 --keep_last_n_checkpoints 3

If GPU memory runs out, lower `batch_size_per_gpu` to 250 and rerun only the final cell. Copy trained checkpoints to your private Drive before the Colab runtime ends; never commit clips or checkpoints to GitHub.